# Phase 6 training attacks on the rs16 models (Colab, browser, ~25 min on a T4)

Attacks `runs/wm_v1/lam20_n256_rs16` seeds 0-2 with the 11 training attacks of the Phase 6 suite (fine-tune x7,
prune + fine-tune x2, overwrite x2) = **33 attacked models**. No watermark key is needed: the attacks only read the
models (already in the public repo); the overwrite attack makes its own attacker key.

**How to run:** colab.research.google.com -> File -> Upload notebook -> this file. Runtime -> Change runtime type ->
**T4 GPU**. Runtime -> **Run all**. At the end `colab_rs16_train.zip` downloads: save it, **not renamed**, into the
repo root on the laptop. The laptop job picks it up by itself, verifies all 33 models and writes the tables.

In [ ]:
# 1. GPU check: stop early if Colab gave a CPU-only runtime
import sys
import torch
print(sys.version.split()[0], "| torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code: branch phase6, without runs/attacks (not needed here, ~200 MB)
import os, pathlib, subprocess

REPO_URL = "https://github.com/shield44-project/post-quantum-cryptography.git"
BRANCH = "phase6"
WORK = "/content/pqw"


def sh(cmd):
    # run a shell command, stream its output, stop Run all if it fails
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError("command failed (exit %d): %s" % (p.returncode, cmd))


if not os.path.isdir(WORK + "/.git"):
    sh(f"git clone --depth 1 --branch {BRANCH} --filter=blob:none --no-checkout {REPO_URL} {WORK}")
    sh(f"git -C {WORK} sparse-checkout set --no-cone '/*' '!/runs/attacks/'")
    sh(f"git -C {WORK} checkout {BRANCH}")
os.chdir(WORK)
sh("git log --oneline -1")
sh("pip install -q 'reedsolo>=1.7'")
os.environ["CIFAR_ROOT"] = "/content/data"
os.environ["PYTHONUNBUFFERED"] = "1"

In [ ]:
# 3. CIFAR-10 to local disk (fails here, not mid-attack, if the download breaks)
import torchvision
for train in (True, False):
    ds = torchvision.datasets.CIFAR10("/content/data", train=train, download=True)
    print("train" if train else "test", len(ds))

In [ ]:
# 4. The 11 training attacks x 3 seeds (finished ones are skipped, so after a disconnect just Run all again)
TAG = "rs16"
SRC = "runs/wm_v1/lam20_n256_rs16/seed{seed}"
OUT = "runs/attacks_" + TAG
sh(f"python -m attacks.make_attacked --cost train --seeds 0 1 2 --src-dir {SRC} --final-from {SRC} "
   f"--out-root {OUT} --data-root /content/data")

In [ ]:
# 5. Zip the 33 attacked models and download (save into the repo root on the laptop, keep the name)
import zipfile
metas = sorted(pathlib.Path(OUT).rglob("attack.json"))
print(len(metas), "attacked models (expected 33)")
assert len(metas) == 33, "some attacks are missing: run cell 4 again"
ZIP = pathlib.Path("/content/colab_%s_train.zip" % TAG)
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(pathlib.Path(OUT).rglob("*")):
        if p.is_file():
            z.write(p, p.as_posix())
print(ZIP, "%.1f MB" % (ZIP.stat().st_size / 1e6))
from google.colab import files
files.download(str(ZIP))